### Задание 1. Реализация AprioriAll
Реализовать алгоритм AprioriAll с нуля на Python.     

Функционал:
- Загрузка данных из CSV (поля: client_id, date, items).
- Преобразование в последовательности клиентов (сортировка по дате,
группировка товаров внутри транзакции).
- Генерация кандидатов и подсчёт поддержки по клиентам.
- Поиск всех частых последовательностей.     

**Проверка**: на синтетическом примере (можно взять пример из лекции с 3–4
клиентами) результат должен совпадать с ручным расчётом.

In [13]:
import pandas as pd
from collections import defaultdict
from io import StringIO

class AprioriAll:
    def __init__(self, min_support=0.5):
        self.min_support = min_support
        self.frequent_sequences = []

    def load_csv(self, csv_str):
        df = pd.read_csv(StringIO(csv_str))
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values(['client_id', 'date'])
        
        sequences = {}
        for client, group in df.groupby('client_id'):
            seq, cur_date, cur_items = [], None, []
            for _, row in group.iterrows():
                items = [i.strip() for i in str(row['items']).split(',')]
                if cur_date is None:
                    cur_date, cur_items = row['date'], items
                elif row['date'] == cur_date:
                    cur_items.extend(items)
                else:
                    seq.append(tuple(sorted(set(cur_items))))
                    cur_date, cur_items = row['date'], items
            if cur_items: seq.append(tuple(sorted(set(cur_items))))
            sequences[client] = tuple(seq)
        return sequences

    def _is_subsequence(self, pattern, sequence):
        p = 0
        for itemset in sequence:
            if p < len(pattern) and set(pattern[p]).issubset(set(itemset)):
                p += 1
        return p == len(pattern)

    def _generate_candidates(self, Lk):
        cands = set()
        for i in range(len(Lk)):
            for j in range(i+1, len(Lk)):
                s1, s2 = Lk[i][0], Lk[j][0]
                # Условие соединения: хвост первой = голова второй
                if s1[1:] == s2[:-1]:
                    cands.add(s1 + (s2[-1],))
        return list(cands)

    def fit(self, sequences):
        n = len(sequences)
        min_cnt = self.min_support * n
        
        # L1: поддержка считается по КЛИЕНТАМ
        item_cnt = defaultdict(int)
        for seq in sequences.values():
            seen = set()
            for itemset in seq:
                for it in itemset:
                    if it not in seen:
                        item_cnt[it] += 1
                        seen.add(it)
                        
        L = []
        for it, cnt in item_cnt.items():
            if cnt >= min_cnt:  # ⚠️ Важно: >=, а не >
                seq = ((it,),)
                L.append((seq, cnt/n))
                self.frequent_sequences.append((seq, cnt/n))
                
        k = 1
        current = L
        while current:
            k += 1
            candidates = self._generate_candidates(current)
            next_L = []
            for cand in candidates:
                sup = sum(1 for s in sequences.values() if self._is_subsequence(cand, s)) / n
                if sup >= self.min_support:
                    next_L.append((cand, sup))
                    self.frequent_sequences.append((cand, sup))
            current = next_L
            
        self.frequent_sequences.sort(key=lambda x: (len(x[0]), -x[1]))
        return self

    def print_results(self):
        print(f"{'Шаблон':<40} | {'Поддержка':<10}")
        print("-" * 55)
        for seq, sup in self.frequent_sequences:
            print(f"{' → '.join(str(s) for s in seq):<40} | {sup:.1%}")

# Ваши данные
csv_data = """client_id,date,items
1,2024-01-01,"milk,bread"
1,2024-01-03,"bread,butter"
1,2024-01-05,"milk,beer"
2,2024-01-02,"milk"
2,2024-01-04,"bread,butter"
2,2024-01-06,"beer"
3,2024-01-01,"milk,bread"
3,2024-01-02,"bread"
3,2024-01-03,"butter"
4,2024-01-01,"beer"
4,2024-01-02,"milk,bread"
4,2024-01-04,"bread,butter"
"""

model = AprioriAll(min_support=0.5)
model.fit(model.load_csv(csv_data))
model.print_results()

### Проверка на синтетическом примере

In [24]:
import pandas as pd
from collections import defaultdict
from io import StringIO

class AprioriAll:
    def __init__(self, min_support=0.5):
        self.min_support = min_support
        self.frequent_sequences = []

    def load_csv(self, csv_str):
        df = pd.read_csv(StringIO(csv_str))
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values(['client_id', 'date'])
        
        sequences = {}
        for client, group in df.groupby('client_id'):
            seq, cur_date, cur_items = [], None, []
            for _, row in group.iterrows():
                items = [i.strip() for i in str(row['items']).split(',')]
                if cur_date is None:
                    cur_date, cur_items = row['date'], items
                elif row['date'] == cur_date:
                    cur_items.extend(items)
                else:
                    seq.append(tuple(sorted(set(cur_items))))
                    cur_date, cur_items = row['date'], items
            if cur_items: seq.append(tuple(sorted(set(cur_items))))
            sequences[client] = tuple(seq)
        return sequences

    def _is_subsequence(self, pattern, sequence):
        p = 0
        for itemset in sequence:
            if p < len(pattern) and set(pattern[p]).issubset(set(itemset)):
                p += 1
        return p == len(pattern)

    def _generate_candidates(self, Lk):
        cands = set()
        for i in range(len(Lk)):
            for j in range(i+1, len(Lk)):
                s1, s2 = Lk[i][0], Lk[j][0]
                # Условие соединения: хвост первой = голова второй
                if s1[1:] == s2[:-1]:
                    cands.add(s1 + (s2[-1],))
        return list(cands)

    def fit(self, sequences):
        n = len(sequences)
        min_cnt = self.min_support * n
        
        # L1: поддержка считается по КЛИЕНТАМ
        item_cnt = defaultdict(int)
        for seq in sequences.values():
            seen = set()
            for itemset in seq:
                for it in itemset:
                    if it not in seen:
                        item_cnt[it] += 1
                        seen.add(it)
                        
        L = []
        for it, cnt in item_cnt.items():
            if cnt >= min_cnt:  # ⚠️ Важно: >=, а не >
                seq = ((it,),)
                L.append((seq, cnt/n))
                self.frequent_sequences.append((seq, cnt/n))
                
        k = 1
        current = L
        while current:
            k += 1
            candidates = self._generate_candidates(current)
            next_L = []
            for cand in candidates:
                sup = sum(1 for s in sequences.values() if self._is_subsequence(cand, s)) / n
                if sup >= self.min_support:
                    next_L.append((cand, sup))
                    self.frequent_sequences.append((cand, sup))
            current = next_L
            
        self.frequent_sequences.sort(key=lambda x: (len(x[0]), -x[1]))
        return self

    def print_results(self):
        print(f"{'Шаблон':<40} | {'Поддержка':<10}")
        print("-" * 55)
        for seq, sup in self.frequent_sequences:
            print(f"{' → '.join(str(s) for s in seq):<40} | {sup:.1%}")

# Ваши данные
csv_data = """client_id,date,items
1,2024-01-01,"milk,bread"
1,2024-01-03,"bread,butter"
1,2024-01-05,"milk,beer"
2,2024-01-02,"milk"
2,2024-01-04,"bread,butter"
2,2024-01-06,"beer"
3,2024-01-01,"milk,bread"
3,2024-01-02,"bread"
3,2024-01-03,"butter"
4,2024-01-01,"beer"
4,2024-01-02,"milk,bread"
4,2024-01-04,"bread,butter"
"""

model = AprioriAll(min_support=0.5)
model.fit(model.load_csv(csv_data))
model.print_results()

Шаблон                                   | Поддержка 
-------------------------------------------------------
('bread',)                               | 100.0%
('milk',)                                | 100.0%
('butter',)                              | 100.0%
('beer',)                                | 75.0%
('milk',) → ('butter',)                  | 100.0%
('bread',) → ('butter',)                 | 75.0%
('butter',) → ('beer',)                  | 50.0%
('bread',) → ('beer',)                   | 50.0%
('milk',) → ('beer',)                    | 50.0%


### Задание 2. Анализ реальных данных

На датасете Online Retail (https://www.kaggle.com/datasets/lakshmi25npathi/onlineretail-dataset/data или аналогичном) выполнить:

1. **Влияние максимальной поддержки**    
Запустить AprioriAll при min_sup = 5%, 10%, 20%.   
Построить график: количество найденных частых последовательностей vs min_sup.    
Сделать вывод о масштабируемости.     

2. **Сравнение с ассоциативными правилами**      
- Применить классический Apriori (без учёта последовательности и ID клиента).
- Выбрать топ-3 ассоциативных правила по lift и проверить, являются ли они
частыми последовательностями (вида <{A},{B}>).
- Объяснить расхождения.

3. **Временные окна**
- Добавить параметр max_gap (максимальный допустимый разрыв между транзакциями одного клиента, например 7 дней).
- Сравнить набор частых последовательностей с max_gap = ∞ и max_gap = 7.
- Привести пример шаблона, который исчезает или появляется.

4. **Визуализация**
- Выбрать один наиболее интересный частый шаблон длины ≥ 2.
- Построить направленный граф перехода товаров.
- Кратко интерпретировать (бизнес-гипотеза).

### Задание 3. Сравнение алгоритмов SPM